<a href="https://colab.research.google.com/github/franz-hufana/Thesis-MobileNet/blob/main/Package-Docs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Thesis System: MobileNetV2 and MobileNetV3 Wood Surface Defect Detection***

Import Necessary Tools

In [ ]:
import os
import gc
import cv2
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from imageio.v2 import imread
from skimage.transform import resize
from skimage.morphology import skeletonize, binary_closing, remove_small_objects, disk
from scipy.ndimage import binary_fill_holes
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2, MobileNetV3Large
from tensorflow.keras import backend as K
from matplotlib.patches import Rectangle
from google.colab import files

print("Import successfully!")

Import successfully!


**Choose among the two options**
1. Import via Google Drive
2. Import via Local Drive

In [ ]:
# Option 1: Google Drive

from google.colab import drive
drive.mount('/content/drive')

# Palitan yung zip_path if iba yung zipfile name
zip_path = "/content/drive/MyDrive/Datasets.zip"

# Kung gusto mo palitan pwede rin
extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Dataset extracted from Drive: {zip_path} to {extract_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset extracted from Drive: /content/drive/MyDrive/Datasets.zip to /content/dataset


In [ ]:
# Option 2: Local Drive or Desktop/Laptop Drive

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
  zip_ref.extractall(extract_path)

print(f"Dataset extracted from upload: {zip_name} to {extract_path}")

**Labels the Datasets:**
* 4 classes in total

In [ ]:
folder_path = "/content/dataset/woods"

label_map = {
    'c' : 0,
    'f' : 1,
    'k' : 2,
    'n' : 3
}

class_names = ['cracks', 'fuzz', 'knothole', 'no defect']
num_classes = len(class_names)
img_size = (224, 224)

print(f"Class Names: {class_names}\nNumber of Classes: {num_classes}\nDataset Folder: {folder_path}")

Class Names: ['cracks', 'fuzz', 'knothole', 'no defect']
Number of Classes: 4
Dataset Folder: /content/dataset/woods


***Data Preprocessing:***
1. Checking and cleaning if its png, jpeg, jpg files and if its not it will be removed
2. Resizing
3. Converting image Grayscale and RGBA to RGB
4. Normalize pixel values to [-1, 1]


In [ ]:
def load_and_preprocess_dataset(folder_path, label_map, img_size=(224, 224)):

    image_paths = []

    for root, dirs, files_in_dir in os.walk(folder_path):
        for file_name in files_in_dir:
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(root, file_name))

    image_paths = sorted(image_paths)
    print(f"Total image files found: {len(image_paths)}")

    data = []
    labels = []
    file_names = []

    for file_path in image_paths:
        file_name = os.path.basename(file_path)
        prefix = file_name[0].lower()

        if prefix not in label_map:
            print(f"Skipping unknown file: {file_name}")
            continue

        try:
            img = imread(file_path)

            # Convert grayscale to RGB
            if img.ndim == 2:
                img = np.stack([img] * 3, axis=-1)

            # Convert RGBA to RGB
            elif img.ndim == 3 and img.shape[-1] == 4:
                img = img[:, :, :3]

            img = resize(
                img,
                img_size,
                anti_aliasing=True,
                preserve_range=True
            ).astype(np.float32)

            # MobileNet normalization to [-1, 1]
            img = (img / 127.5) - 1.0

            data.append(img)
            labels.append(label_map[prefix])
            file_names.append(file_name)

        except Exception as e:
            print(f"Error reading {file_name}: {e}")

    data = np.array(data, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)

    print("\nData Shape:", data.shape)
    print("Labels Shape:", labels.shape)

    print("\nSample Loaded Files:")
    for i in range(min(10, len(file_names))):
        print(f"{file_names[i]} ---> {class_names[labels[i]]}")

    return data, labels, file_names


data, labels, file_names = load_and_preprocess_dataset(
    folder_path=folder_path,
    label_map=label_map,
    img_size=img_size
)

Total image files found: 4000

Data Shape: (4000, 224, 224, 3)
Labels Shape: (4000,)

Sample Loaded Files:
c(1).png ---> cracks
c(10).png ---> cracks
c(100).png ---> cracks
c(101).png ---> cracks
c(102).png ---> cracks
c(103).png ---> cracks
c(104).png ---> cracks
c(105).png ---> cracks
c(106).png ---> cracks
c(107).png ---> cracks


***Splitting the Dataset for Training and Validating:***

* Can change the parameter if not satisfied to the split and results of training

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    data,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print("Training Data Shape :", x_train.shape)
print("Training Labels Shape:", y_train.shape)
print("Validation Data Shape:", x_val.shape)
print("Validation Labels Shape:", y_val.shape)

***Class Weights:***

* For Class or Dataset Imbalance



In [ ]:
unique_classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train
)
class_weight_dict = dict(zip(unique_classes, class_weights_array))

print("Class Weights:")
for k, v in class_weight_dict.items():
    print(f"{class_names[k]}: {v:.4f}")

***Creating Data Augmentation Layer:***
* Add more Layers if applicable

In [ ]:
def get_data_augmentation():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.15),
    ], name="data_augmentation")

***Model Section:***
1. Preloading the Pretrained Models
2. Creating New Classifier head for transfer learning stage
3. Compile the New Classifier head

In [ ]:
def build_model(backbone_name, input_shape=(224, 224, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape, name="input_image")
    augmentation = get_data_augmentation()

    if backbone_name == "MobileNetV2":
        base_model = MobileNetV2(
            input_shape=input_shape,
            include_top=False,
            weights="imagenet"
        )

    elif backbone_name == "MobileNetV3":
        base_model = MobileNetV3Large(
            input_shape=input_shape,
            include_top=False,
            weights="imagenet",
            include_preprocessing=False
        )

    else:
        raise ValueError("Unsupported model name")

    # Freeze base model for transfer learning stage
    base_model.trainable = False

    x = augmentation(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(128, activation="relu", name="dense_128")(x)
    x = layers.Dropout(0.4, name="dropout")(x)
    outputs = Dense(num_classes, activation="softmax", name="wood_output")(x)

    model = Model(inputs=inputs, outputs=outputs, name=f"{backbone_name}_wood_model")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"]
    )

    return model

***Optional Step:***
* Fine-Tuning Function by unfreezing the last layers after initial training

In [ ]:
def fine_tune_model(model, backbone_name, unfreeze_last_layers=30, fine_tune_lr=1e-5):
    if backbone_name == "MobileNetV2":
        base_model_name = "mobilenetv2_1.00_224"
    elif backbone_name == "MobileNetV3":
        base_model_name = "MobilenetV3large"
    else:
        raise ValueError("Unsupported model")

    # Find the base model dynamically
    base_model = None
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            base_model = layer
            break

    if base_model is None:
        print("Base model not found. Fine-tuning skipped.")
        return model

    base_model.trainable = True

    for layer in base_model.layers[:-unfreeze_last_layers]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"]
    )

    print(f"\nFine-tuning enabled for {backbone_name}")
    print(f"Last {unfreeze_last_layers} layers are trainable.")
    return model

***Model's Train and evaluate Section***

In [ ]:
def train_and_evaluate(
    backbone_name,
    x_train, y_train,
    x_val, y_val,
    class_names,
    class_weight_dict,
    epochs=20,
    batch_size=32,
    do_fine_tune=False,

    # Optional or removed (#) to use
    # fine_tune_epochs=10
):
    print("\n" + "=" * 70)
    print(f"TRAINING {backbone_name}")
    print("=" * 70)

    K.clear_session()
    gc.collect()

    model = build_model(
        backbone_name=backbone_name,
        input_shape=(224, 224, 3),
        num_classes=len(class_names)
    )

    print(f"\nMODEL SUMMARY: {backbone_name}")
    model.summary()


# Frozen Base Model
    history_stage1 = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        shuffle=True,
        verbose=2,
        class_weight=class_weight_dict
    )

    combined_history = {
        "accuracy": list(history_stage1.history["accuracy"]),
        "val_accuracy": list(history_stage1.history["val_accuracy"]),
        "loss": list(history_stage1.history["loss"]),
        "val_loss": list(history_stage1.history["val_loss"])
    }





***NOTE:***
* Only Use this if using Fine-tuning setting

In [ ]:
# Use if Fine-tuning is available
if do_fine_tune:
        model = fine_tune_model(
            model=model,
            backbone_name=backbone_name,
            unfreeze_last_layers=30,
            fine_tune_lr=1e-5
        )

        history_stage2 = model.fit(
            x_train, y_train,
            validation_data=(x_val, y_val),
            epochs=fine_tune_epochs,
            batch_size=batch_size,
            shuffle=True,
            verbose=2,
            class_weight=class_weight_dict
        )

        combined_history["accuracy"] += list(history_stage2.history["accuracy"])
        combined_history["val_accuracy"] += list(history_stage2.history["val_accuracy"])
        combined_history["loss"] += list(history_stage2.history["loss"])
        combined_history["val_loss"] += list(history_stage2.history["val_loss"])

***Validate Prediction of the Model***

In [ ]:
# Validation Prediction

    val_probs = model.predict(x_val, batch_size=batch_size, verbose=0)
    val_preds = np.argmax(val_probs, axis=1)

    cm = confusion_matrix(y_val, val_preds)

    print(f"\nCLASSIFICATION REPORT: {backbone_name}")
    print(classification_report(y_val, val_preds, target_names=class_names))

    macro_f1 = f1_score(y_val, val_preds, average='macro')
    micro_f1 = f1_score(y_val, val_preds, average='micro')
    weighted_f1 = f1_score(y_val, val_preds, average='weighted')

    metrics_dict = {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "weighted_f1": weighted_f1,
        "val_accuracy_last": combined_history["val_accuracy"][-1],
        "val_loss_last": combined_history["val_loss"][-1]
    }

    return model, combined_history, cm, metrics_dict

***Finalize the Model Before Training***
* Puts Everything together

In [ ]:
mobilenetv2_model, history_v2, cm_v2, metrics_v2 = train_and_evaluate(
    backbone_name="MobileNetV2",
    x_train=x_train, y_train=y_train,
    x_val=x_val, y_val=y_val,
    class_names=class_names,
    class_weight_dict=class_weight_dict,
    epochs=20,
    batch_size=32,
    do_fine_tune=True,
    fine_tune_epochs=10
)

mobilenetv3_model, history_v3, cm_v3, metrics_v3 = train_and_evaluate(
    backbone_name="MobileNetV3",
    x_train=x_train, y_train=y_train,
    x_val=x_val, y_val=y_val,
    class_names=class_names,
    class_weight_dict=class_weight_dict,
    epochs=20,
    batch_size=32,
    do_fine_tune=True,
    fine_tune_epochs=10
)

***Model's Performance***
* Learning Curves
* Confusion Matrices
* F1-scores

In [ ]:
# Learning Curves

def plot_learning_curves(history, title):
    plt.figure(figsize=(10, 5))
    plt.plot(history["accuracy"], label='Train Accuracy')
    plt.plot(history["val_accuracy"], label='Validation Accuracy')
    plt.plot(history["loss"], label='Train Loss')
    plt.plot(history["val_loss"], label='Validation Loss')
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy / Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_learning_curves(history_v2, "MobileNetV2 Training and Validation Performance")
plot_learning_curves(history_v3, "MobileNetV3Large Training and Validation Performance")

In [ ]:
# Confusion Matrix

def plot_confusion_matrix(cm, class_names, title):
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

plot_confusion_matrix(cm_v2, class_names, "MobileNetV2 Confusion Matrix")
plot_confusion_matrix(cm_v3, class_names, "MobileNetV3Large Confusion Matrix")

In [ ]:
print("\nMobileNetV2")
for k, v in metrics_v2.items():
    print(f"{k}: {v:.4f}")

print("\nMobileNetV3Large")
for k, v in metrics_v3.items():
    print(f"{k}: {v:.4f}")

In [ ]:
# =========================================================
# 15. GRAD-CAM UTILITIES
# Improved localization instead of whole-image boxes
# =========================================================
def preprocess_single_image_for_model(img, target_size=(224, 224)):
    img_resized = resize(
        img,
        target_size,
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

    img_resized = (img_resized / 127.5) - 1.0
    return np.expand_dims(img_resized, axis=0)


def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        try:
            if len(layer.output.shape) == 4:
                return layer.name
        except:
            continue
    raise ValueError("No convolutional layer found.")


def make_gradcam_heatmap(img_array, model, pred_index=None, last_conv_layer_name=None):
    if last_conv_layer_name is None:
        last_conv_layer_name = find_last_conv_layer(model)

    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index), predictions.numpy()[0]


# =========================================================
# 16. HEATMAP REFINEMENT
# =========================================================
def heatmap_to_binary_mask(heatmap, original_shape, threshold=0.45):
    h, w = original_shape[:2]
    heatmap_resized = cv2.resize(heatmap, (w, h))

    mask = heatmap_resized >= threshold
    mask = binary_closing(mask, disk(3))
    mask = binary_fill_holes(mask)
    mask = remove_small_objects(mask, min_size=80)

    return mask.astype(np.uint8), heatmap_resized


# =========================================================
# 17. CRACK TRACING
# For cracks, draw line-like defect pattern instead of one big box
# =========================================================
def refine_crack_mask(image_rgb, coarse_mask):
    gray = cv2.cvtColor(image_rgb.astype(np.uint8), cv2.COLOR_RGB2GRAY)

    masked_gray = gray.copy()
    masked_gray[coarse_mask == 0] = 0

    edges = cv2.Canny(masked_gray, 40, 120)

    crack_mask = (edges > 0) & (coarse_mask > 0)
    crack_mask = binary_closing(crack_mask, disk(1))
    crack_mask = remove_small_objects(crack_mask, min_size=30)

    crack_skeleton = skeletonize(crack_mask)
    return crack_skeleton.astype(np.uint8)


# =========================================================
# 18. REGION BOXES FOR FUZZ AND KNOTHOLE
# =========================================================
def get_boxes_from_mask(mask, min_area=100):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8),
        connectivity=8
    )

    boxes = []
    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        if area >= min_area:
            boxes.append((x, y, x + w, y + h, area))
    return boxes


# =========================================================
# 19. VISUALIZATION
# =========================================================
def visualize_localization(image, pred_class_name, confidence, heatmap, crack_mask=None, boxes=None):
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)

    # Heatmap overlay
    ax.imshow(heatmap, cmap='jet', alpha=0.35)

    # Region boxes for fuzz / knothole
    if boxes is not None:
        for (x1, y1, x2, y2, area) in boxes:
            rect = Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor='yellow',
                facecolor='none'
            )
            ax.add_patch(rect)

    # Crack tracing
    if crack_mask is not None:
        ys, xs = np.where(crack_mask > 0)
        ax.scatter(xs, ys, s=1, c='cyan')

    ax.set_title(f"Prediction: {pred_class_name} | Confidence: {confidence:.3f}")
    ax.axis("off")
    plt.show()


# =========================================================
# 20. SINGLE IMAGE DEFECT LOCALIZATION
# =========================================================
def localize_defect(image_path, model, class_names, heatmap_threshold=0.45):
    img = imread(image_path)

    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    elif img.ndim == 3 and img.shape[-1] == 4:
        img = img[:, :, :3]

    img = img.astype(np.uint8)

    img_input = preprocess_single_image_for_model(img)

    heatmap, pred_idx, probs = make_gradcam_heatmap(img_input, model)
    pred_class = class_names[pred_idx]
    confidence = float(probs[pred_idx])

    coarse_mask, heatmap_resized = heatmap_to_binary_mask(
        heatmap=heatmap,
        original_shape=img.shape,
        threshold=heatmap_threshold
    )

    crack_mask = None
    boxes = None

    if pred_class == "cracks":
        crack_mask = refine_crack_mask(img, coarse_mask)

    elif pred_class in ["fuzz", "knothole"]:
        boxes = get_boxes_from_mask(coarse_mask, min_area=100)

    visualize_localization(
        image=img,
        pred_class_name=pred_class,
        confidence=confidence,
        heatmap=heatmap_resized,
        crack_mask=crack_mask,
        boxes=boxes
    )

    print("\nPrediction Probabilities:")
    for i, cls in enumerate(class_names):
        print(f"{cls}: {probs[i]:.4f}")

    return {
        "predicted_class": pred_class,
        "confidence": confidence,
        "probabilities": probs,
        "coarse_mask": coarse_mask,
        "crack_mask": crack_mask,
        "boxes": boxes
    }


# =========================================================
# 21. TEST LOCALIZATION ON A NEW IMAGE
# =========================================================
print("\n=========================================================")
print("UPLOAD ONE TEST IMAGE FOR DEFECT LOCALIZATION")
print("=========================================================")

uploaded_test = files.upload()
test_image_name = list(uploaded_test.keys())[0]

print("\n--- MobileNetV2 Localization ---")
result_v2 = localize_defect(
    image_path=test_image_name,
    model=mobilenetv2_model,
    class_names=class_names,
    heatmap_threshold=0.45
)

print("\n--- MobileNetV3Large Localization ---")
result_v3 = localize_defect(
    image_path=test_image_name,
    model=mobilenetv3_model,
    class_names=class_names,
    heatmap_threshold=0.45
)


# =========================================================
# 22. HARD-NEGATIVE MINING HELPER
# This helps reduce confusion between natural wood grain and defects
# =========================================================
def extract_false_positive_patches(
    clean_image_path,
    model,
    save_dir="hard_negative_patches",
    window_size=224,
    step=112,
    conf_threshold=0.75
):
    """
    Use this only on clean wood images (true no-defect images).
    Any patch predicted as crack/fuzz/knothole is saved as a hard negative.
    These patches can later be added back to the 'no defect' class.
    """

    os.makedirs(save_dir, exist_ok=True)

    img = imread(clean_image_path)

    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    elif img.ndim == 3 and img.shape[-1] == 4:
        img = img[:, :, :3]

    img = img.astype(np.uint8)
    h, w = img.shape[:2]

    saved_count = 0

    for y in range(0, h - window_size + 1, step):
        for x in range(0, w - window_size + 1, step):
            patch = img[y:y + window_size, x:x + window_size]

            patch_input = preprocess_single_image_for_model(patch)
            probs = model.predict(patch_input, verbose=0)[0]

            pred_idx = np.argmax(probs)
            pred_class = class_names[pred_idx]
            confidence = float(probs[pred_idx])

            if pred_class != "no defect" and confidence >= conf_threshold:
                save_name = f"n_hardneg_{saved_count}_{pred_class}_{confidence:.2f}.jpg"
                save_path = os.path.join(save_dir, save_name)

                cv2.imwrite(save_path, cv2.cvtColor(patch, cv2.COLOR_RGB2BGR))
                saved_count += 1

    print(f"Hard negative patches saved: {saved_count}")
    print(f"Saved to folder: {save_dir}")


# =========================================================
# 23. SAVE TRAINED MODELS
# =========================================================
mobilenetv2_model.save("wood_defect_mobilenetv2_system.h5")
mobilenetv3_model.save("wood_defect_mobilenetv3large_system.h5")

print("\nModels saved successfully:")
print(" - wood_defect_mobilenetv2_system.h5")
print(" - wood_defect_mobilenetv3large_system.h5")


# =========================================================
# 24. PANEL-FRIENDLY SUMMARY
# =========================================================
print("\n=========================================================")
print("IMPORTANT SYSTEM NOTE")
print("=========================================================")
print("1. The classification stage identifies the defect class of the wood surface.")
print("2. The localization stage uses Grad-CAM to identify the most important defect region.")
print("3. For region-like defects such as fuzz and knothole, bounding boxes are used.")
print("4. For line-like defects such as cracks, crack tracing is applied instead of one large box.")
print("5. Hard-negative mining can be used to reduce confusion between natural wood grain and true defects.")